# 04 · Tools & Identity — give the agent hands (safely)
### Code Interpreter · Browser · Gateway · Identity

A model that can only talk is a chatbot. Tools make it an agent. AgentCore gives you three managed tool surfaces plus the identity layer that governs all of them:

- **Code Interpreter** — a sandboxed runtime for *exact* computation (the model writes/runs code instead of guessing arithmetic).
- **Browser** — a managed headless browser for live web when there's no API.
- **Gateway** — turns your **existing** Lambdas / OpenAPI / Smithy services into **MCP tools** the agent can call, with auth and governance — no agent rewrite.
- **Identity** — *inbound*: who's allowed to invoke the agent. *outbound*: how the agent authenticates to external APIs without you hardcoding secrets.

```mermaid
flowchart LR
    AG[Agent] --> CI[Code Interpreter<br/>sandboxed compute]
    AG --> BR[Browser<br/>live web]
    AG --> GW[Gateway<br/>your APIs as MCP tools]
    ID[Identity] -. who can call .-> AG
    ID -. how to call out .-> GW
```

> Prereqs: notebook 00. Code Interpreter and Browser spin up **managed sandbox sessions** (small cost). Gateway and Identity create resources you'll usually set up once via console/IaC — we show the verified API calls and the runnable connection pattern.


In [ ]:
import os, json, boto3
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
REGION = "us-east-1"
MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
from strands import Agent, tool
print("ready")

## 1. Code Interpreter — stop the model from doing mental math

LLMs are unreliable at arithmetic. The Code Interpreter tool lets the agent **write and execute Python in a sandbox**, then read the result back — so numbers are computed, not hallucinated. Perfect for fare math, EU261 compensation, currency conversion, date diffs.

`AgentCoreCodeInterpreter(region=...)` exposes a tool at `.code_interpreter`. Hand that to the agent — `auto_create` spins up the sandbox on first use.


In [ ]:
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

ci = AgentCoreCodeInterpreter(region=REGION)   # manages a sandboxed session for you

calc_agent = Agent(
    model=MODEL_HAIKU,
    tools=[ci.code_interpreter],               # .code_interpreter is the tool object
    system_prompt="You are TravelMind. For any calculation, run code to get the exact value, then state it plainly.",
)

ans = calc_agent(
    "A passenger is owed EU261 compensation of 600 EUR. Convert to GBP at 0.85, "
    "add a 25 GBP meal voucher, then split the total across 3 travelers. Give the per-person amount."
)
print(str(ans))

Behind that answer, the agent generated Python, ran it in an isolated sandbox, and reported the computed number. No silent arithmetic errors. The same sandbox can read/write files and run multi-step scripts — useful for parsing an itinerary CSV or generating a chart.


## 2. Browser — live web when there's no API

Some facts only live on a web page (an airline's live status wording, a visa rule page). The Browser tool drives a **managed headless browser** so the agent can navigate and read.

`AgentCoreBrowser(region=...)` exposes a tool at `.browser`. It uses Playwright locally, so install it first:

```bash
pip install playwright nest_asyncio
playwright install chromium
```


In [ ]:
# Local runtime deps for the browser tool:
# !pip install -q playwright nest_asyncio && playwright install chromium
import nest_asyncio
nest_asyncio.apply()   # lets the async browser run inside the notebook's event loop

from strands_tools.browser import AgentCoreBrowser

br = AgentCoreBrowser(region=REGION)

web_agent = Agent(
    model=MODEL_HAIKU,
    tools=[br.browser],   # .browser is the tool object
    system_prompt="You are TravelMind. Use the browser to check live public info when no API exists, then summarize briefly.",
)

print(str(web_agent("Open https://www.heathrow.com and tell me what top-level travel info the homepage surfaces.")))

**Skeptic's note:** browsing is powerful but slow and brittle vs an API. Reach for Browser only when there's genuinely no structured endpoint — and prefer Gateway (next) to wrap a real API whenever one exists.


## 3. Gateway — your existing APIs as governed MCP tools

You already have a flight API (a Lambda, or an OpenAPI service). You don't want to rewrite it as Strands `@tool` functions, copy auth into the agent, or re-implement it per framework. **Gateway** fronts those backends and exposes them as **MCP tools** with centralized auth — the agent just connects to one MCP endpoint and gets all the tools.

```mermaid
flowchart LR
    L[Your Lambda / OpenAPI service] --> T[Gateway Target]
    T --> G[Gateway<br/>MCP endpoint + auth]
    G --> A[Any agent / framework<br/>connects via MCP]
```

### 3a) Create the gateway + a target (done once; console or IaC is common)


In [ ]:
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)

# A gateway speaks MCP and is protected by an authorizer (here: a JWT issuer / your IdP).
gw = ctrl.create_gateway(
    name="travelmind-gateway",
    roleArn="<gateway-execution-role-arn>",                 # role the gateway assumes to reach targets
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": "https://<your-idp>/.well-known/openid-configuration",
        "allowedClients": ["<your-client-id>"],
    }},
)
GATEWAY_URL = gw["gatewayUrl"]        # <-- the MCP endpoint agents connect to
print("gateway:", gw["gatewayId"], GATEWAY_URL)

# A target attaches a backend. targetConfiguration is {'mcp': {...}} for a Lambda/OpenAPI,
# or {'http': {...}}. The exact inner schema is your backend's shape — fill from the
# Gateway target docs (Lambda ARN or OpenAPI document) plus its credential provider.
ctrl.create_gateway_target(
    gatewayIdentifier=gw["gatewayId"],
    name="flight-api",
    targetConfiguration={"mcp": {"...": "your Lambda ARN or OpenAPI schema here"}},
    credentialProviderConfigurations=[{"...": "how the gateway authenticates to your backend"}],
)

### 3b) Connect an agent to the gateway (this part is fully runnable)

The agent treats the gateway as an **MCP server**. Strands' `MCPClient` opens a connection; `list_tools_sync()` pulls the gateway's tools; you pass them straight into `Agent(tools=...)`. Auth is a **bearer token** from your IdP in the request header.


In [ ]:
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

BEARER = "<jwt-or-oauth-access-token-from-your-idp>"

# transport_callable: a zero-arg function returning the MCP transport (streamable HTTP + auth header)
gateway_client = MCPClient(
    lambda: streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {BEARER}"})
)

with gateway_client:                                   # MCPClient is a context manager
    tools = gateway_client.list_tools_sync()           # the gateway's tools, as MCP tools
    gw_agent = Agent(model=MODEL_HAIKU, tools=tools,
                     system_prompt="You are TravelMind. Use the gateway tools to answer.")
    print(str(gw_agent("Look up the status of flight BA117 using the available tools.")))

## 4. Identity — who can call in, and how the agent calls out

Two directions, don't conflate them:

**Inbound (authn/authz to invoke your agent).** By default, invoking a Runtime agent requires AWS IAM. For human or third-party callers you attach a **JWT/OAuth authorizer** so only valid tokens get through. With the toolkit/CLI you set this via the runtime's `authorizer_configuration` (the same `customJWTAuthorizer` shape Gateway uses): a `discoveryUrl` + allowed clients.

**Outbound (the agent authenticating to external APIs).** You don't bake the Aviationstack key or a partner's OAuth into your code. You register a **credential provider** once, and AgentCore Identity hands the agent a fresh credential at call time from its token vault.


In [ ]:
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)

# Register an API-key provider once (e.g. the flight data API key lives in the vault, not your code):
ctrl.create_api_key_credential_provider(
    name="aviationstack",
    apiKey="<your-aviationstack-api-key>",
)

# Or an OAuth2 provider for a partner API:
# ctrl.create_oauth2_credential_provider(
#     name="partner-air",
#     credentialProviderVendor="CustomOauth2",
#     oauth2ProviderConfigInput={ "...": "issuer, client id/secret, token url — per the vendor schema" },
# )
print("credential provider registered")

### The ergonomic way to use outbound creds: decorators

`bedrock_agentcore.identity` gives you `@requires_api_key` and `@requires_access_token`. Decorate a function and AgentCore injects a fresh credential as a keyword arg — your code never sees the secret at rest. (These wrap async flows, so the function is `async`.)


In [ ]:
from bedrock_agentcore.identity import requires_api_key
import urllib.request

@requires_api_key(provider_name="aviationstack", into="api_key")   # 'api_key' is injected for you
async def fetch_flight(flight_number: str, *, api_key: str) -> dict:
    """Call the real flight API using a key supplied by AgentCore Identity at runtime."""
    url = f"https://api.aviationstack.com/v1/flights?access_key={api_key}&flight_iata={flight_number}"
    with urllib.request.urlopen(url, timeout=15) as r:
        return json.loads(r.read())

# In an agent, wrap it as a @tool that awaits this. Inside AgentCore Runtime the injection
# resolves against the deployed identity; locally you'd supply a token via the workload context.

**Under the hood**, the decorators call the data-plane Identity APIs you can also call directly:

- `get_workload_access_token(workloadName=...)` → `workloadAccessToken` (proves *which agent* is asking).
- `get_resource_api_key(workloadIdentityToken=..., resourceCredentialProviderName="aviationstack")` → `apiKey`.
- `get_resource_oauth2_token(workloadIdentityToken=..., resourceCredentialProviderName=..., scopes=[...], oauth2Flow="M2M")` → `accessToken`.

The decorator just makes that flow disappear.


## Common errors → fixes

| Symptom | Cause | Fix |
|---|---|---|
| Code Interpreter / Browser `AccessDenied` | Principal can't create the sandbox | Ensure AgentCore tool permissions on your role |
| Browser cell hangs / event-loop error | Missing `nest_asyncio` or Playwright | `pip install playwright nest_asyncio` + `playwright install chromium`, call `nest_asyncio.apply()` |
| `MCPClient` connects but lists no tools | Gateway target not synced / bad token | Confirm the target's status is ready and the bearer token is valid for `allowedClients` |
| 401/403 invoking the agent | Inbound authorizer rejected the token | Token must match the runtime's `discoveryUrl` + allowed clients/audience |
| Secret visible in logs | Hardcoded API key | Use a credential provider + `@requires_api_key`; never log the injected value |

> **Production note:** scope the gateway/runtime execution roles to least privilege, keep all third-party secrets in Identity (never in code or env files you commit), put a JWT authorizer in front of any externally-reachable agent, and prefer Gateway over re-implementing tools so auth and rate-limiting stay centralized.

## Next

**`05_multi_agent_orchestration.ipynb`** — one agent isn't enough for TravelMind. Build **every** multi-agent pattern: agents-as-tools (hierarchical), Swarm (autonomous handoff), Graph (deterministic + conditional + parallel), sequential workflow, and the A2A protocol for agents that live in different processes/runtimes.
